In [37]:
# Load in merged data frame 
# copied and pasted from Data/Data Scripts/merge_station_and_event_data_script.py

import pandas as pd
import numpy as np
import os
import math

# What this does: Counts the frequency of non nan's for each feature for each station
# and organizes this information into Features_No_NAN_Counts.csv.

# Note: In the documentation (isd-format-document.pdf) sometimes a number 
# stands in for a nan value. We will deal with that if we choose such a 
# feature later on.


dirc = 'Data/OK City Station Data/Raw Data'
raw_station_data = os.listdir('Data/OK City Station Data/Raw Data') # list files in directory

station_csv_files = [file for file in raw_station_data if ('.csv' in file) and ('Station' in file)] # get csv files

station_list = [] # to hold station numbers as keys and number of csv files as entries

for file in station_csv_files:
    underscore_split = file.split('_')
    station_num = underscore_split[1].replace('.csv','')  
    station_list.append(int(station_num)) # station_num is a string
    


# Read in CSV's as pd.DataFrame's concat along axis = 0 

station_pd_dfs=[pd.read_csv(f"{dirc}/{file}").copy() for file in station_csv_files]
stations_df = pd.concat(station_pd_dfs, axis = 0)
new_df = stations_df.reset_index().drop(['index', 'Unnamed: 0'], axis =1).copy()



new_df['YEAR-MONTH-DAY'] = new_df['DATE'].apply(lambda r: r.split('T')[0])
new_df['TIME'] = new_df['DATE'].apply(lambda r: r.split('T')[1])
new_df.drop(['DATE'],axis=1,inplace=True)

# combined oklahoma tornadoes

path = 'Data/Storm Event Data/Cleaned Data/NEW_Oklahoma_Tornadoes_2000_2021.csv'
df2 = pd.read_csv(path).copy()
# Really will only care about 'BEGIN_DATE_TIME', 'END_DATE_TIME', 'BEGIN_LAT',
# 'END_LAT', 'BEGIN_LON', 'END_LON'

df2 = df2[
        [
        'BEGIN_DATE_TIME', 
        'END_DATE_TIME', 
        'BEGIN_LAT', 
        'END_LAT', 
        'BEGIN_LON', 
        'END_LON'
        ]
        ]

# columns BEGIN-DATE and BEGIN-TIME
new_df2 = df2.copy()
new_df2['BEGIN_DATE'] = df2['BEGIN_DATE_TIME'].apply(lambda r : r.split(' ')[0])
new_df2['BEGIN_TIME'] = df2['BEGIN_DATE_TIME'].apply(lambda r : r.split(' ')[1])
new_df2['END_DATE'] = df2['END_DATE_TIME'].apply(lambda r : r.split(' ')[0])
new_df2['END_TIME'] = df2['END_DATE_TIME'].apply(lambda r : r.split(' ')[1])

new_df2.drop(['BEGIN_DATE_TIME','END_DATE_TIME'],axis=1,inplace=True)

# Change format of 'BEGIN_DATE' and 'END_DATE' to match that of 'YEAR_MONTH_DAY' in station data

month_num = {
            'JAN':'01',
            'FEB':'02',
            'MAR':'03', 
            'APR':'04', 
            'MAY':'05', 
            'JUN':'06', 
            'JUL':'07', 
            'AUG':'08', 
            'SEP':'09', 
            'OCT':'10', 
            'NOV':'11', 
            'DEC':'12'
            }

new_df2['BEGIN_DATE'] = new_df2['BEGIN_DATE'].apply(lambda r: f'{r.split('-')[0]}-{month_num[r.split('-')[1]]}-{r.split('-')[2]}')
new_df2['END_DATE'] = new_df2['END_DATE'].apply(lambda r: f'{r.split('-')[0]}-{month_num[r.split('-')[1]]}-{r.split('-')[2]}')

# Get year to be 20**
new_df2['BEGIN_DATE']=new_df2['BEGIN_DATE'].apply(lambda r: f'20{r.split('-')[2]}-{r.split('-')[1]}-{r.split('-')[0]}')
new_df2['END_DATE']=new_df2['END_DATE'].apply(lambda r: f'20{r.split('-')[2]}-{r.split('-')[1]}-{r.split('-')[0]}')

data = pd.merge(left=new_df,right=new_df2,how ='outer',left_on='YEAR-MONTH-DAY',right_on='BEGIN_DATE')

/var/folders/_y/61ngw3jd5739nsvht6fg3wfr0000gn/T/ipykernel_22348/1703931115.py:33: DtypeWarning: Columns (7,14,15,16,17,19,20,21,22,23,24,25,26,27,28,29,30,32,33,34,40,41,42,43,44,45,46,47,50,51,52,56,57,58,59,60,65,68,69,70,71,76,79,80,81,82,83,90,91,92,93,94,95,96,97,98,99,102,104,105,106,110,111,113,120,121,122,125) have mixed types. Specify dtype option on import or set low_memory=False.
  station_pd_dfs=[pd.read_csv(f"{dirc}/{file}").copy() for file in station_csv_files]
/var/folders/_y/61ngw3jd5739nsvht6fg3wfr0000gn/T/ipykernel_22348/1703931115.py:33: DtypeWarning: Columns (39,40,41,42,43,47,48,52,53,54,55,57,58,59,60,61,65,70,71,76,77,88,89,106,108,109,110) have mixed types. Specify dtype option on import or set low_memory=False.
  station_pd_dfs=[pd.read_csv(f"{dirc}/{file}").copy() for file in station_csv_files]
/var/folders/_y/61ngw3jd5739nsvht6fg3wfr0000gn/T/ipykernel_22348/1703931115.py:33: DtypeWarning: Columns (15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,32,34,40,41,4

,STATION,NAME,LATITUDE,LONGITUDE,ELEVATION,SOURCE,REPORT_TYPE,CALL_SIGN,QUALITY_CONTROL,AA1,...,YEAR-MONTH-DAY,TIME,BEGIN_LAT,END_LAT,BEGIN_LON,END_LON,BEGIN_DATE,BEGIN_TIME,END_DATE,END_TIME
0,72353013967,"OKLAHOMA CITY WILL ROGERS WORLD AIRPORT, OK US",35.38890,-97.60060,391.7,3,SY-MT,OKC,V020,"01,0000,9,5",...,2000-01-01,00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,72353013967,"OKLAHOMA CITY WILL ROGERS WORLD AIRPORT, OK US",35.38890,-97.60060,391.7,3,FM-15,OKC,V020,"01,0000,9,5",...,2000-01-01,01:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,72353013967,"OKLAHOMA CITY WILL ROGERS WORLD AIRPORT, OK US",35.38890,-97.60060,391.7,3,FM-15,OKC,V020,"01,0000,9,5",...,2000-01-01,02:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,72353013967,"OKLAHOMA CITY WILL ROGERS WORLD AIRPORT, OK US",35.38890,-97.60060,391.7,3,FM-15,OKC,V020,"01,0000,9,5",...,2000-01-01,03:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,72353013967,"OKLAHOMA CITY WILL ROGERS WORLD AIRPORT, OK US",35.38890,-97.60060,391.7,3,FM-15,OKC,V020,"01,0000,9,5",...,2000-01-01,04:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1626347,72354403954,"OKLAHOMA CITY WILEY POST AIRPORT, OK US",35.54113,-97.64725,390.2,7,FM-15,KPWA,V020,"01,0000,9,5",...,2021-12-31,19:53:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1626348,72354403954,"OKLAHOMA CITY WILEY POST AIRPORT, OK US",35.54113,-97.64725,390.2,7,FM-15,KPWA,V020,"01,0000,9,5",...,2021-12-31,20:53:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1626349,72354403954,"OKLAHOMA CITY WILEY POST AIRPORT, OK US",35.54113,-97.64725,390.2,7,FM-15,KPWA,V020,"01,0000,9,5",...,2021-12-31,21:53:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1626350,72354403954,"OKLAHOMA CITY WILEY POST AIRPORT, OK US",35.54113,-97.64725,390.2,7,FM-15,KPWA,V020,"01,0000,9,5",...,2021-12-31,22:53:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [39]:
def lat_lon_metric(lat_lon_1:tuple[float,float],lat_lon_2:tuple[float,float]):
    # Haversine Formula for computing the number of kilometers between 
    # To use this must convert lat lon from degrees to radians.
    Rd = 6371 # approx. Earth radius in km
    lat_1,lon_1 = lat_lon_1[0],lat_lon_1[1] # in deg
    lat_2,lon_2 = lat_lon_2[0],lat_lon_2[1] # in deg
    rlat_1 = (lat_1*(math.pi))/(180) # convert to radians
    rlon_1 = (lon_1*math.pi)/(180) # convert to radians
    rlat_2 = (lat_2*math.pi)/(180) # convert to radians
    rlon_2 = (lon_2*math.pi)/(180) # convert to radians
    Term_1 = (math.sin((rlat_2-rlat_1)/2))**2
    Term_2 = math.cos(rlat_1)*math.cos(rlat_2)*(math.sin((rlon_2-rlon_1)/2))**2
    a = Term_1+Term_2
    c = 2*math.atan2(math.sqrt(a),math.sqrt(1-a))
    D=Rd*c
    return D

def within_radius(lat_lon_1:tuple[float,float],lat_lon_2:tuple[float,float], valid_radius :float):
    if np.isnan(lat_lon_1[0]):
        return np.nan
    if np.isnan(lat_lon_1[0]):
        return np.nan
    if np.isnan(lat_lon_1[0]):
        return np.nan
    if np.isnan(lat_lon_1[0]):
        return np.nan
    
    great_circle_distance = lat_lon_metric(lat_lon_1,lat_lon_2)
    
    if great_circle_distance <= valid_radius:
        return True
    else:
        return False

In [ ]:
# lines up with https://www.omnicalculator.com/other/latitude-longitude-distance
# example with Paris (48.8566,2.3522) and Krakow (50.0647,19.9450)
# Uses atan2 for more accuracy.
lat_lon_metric((48.8566,2.3522),(50.0647,19.9450)) 

1275.5699719152508

In [40]:
# Create Tornado indicator column
# Things to keep in mind:
# 1. At what time should a tornado be indicated? 
#       eg. during which time periods should we indicate? The same day? within n hours?
# 2. How far from the station should a tornado be in order to count for that station?


### HYPERPARAMETER time_window in hours
### if a tornado happens at 16:00 and time_window is 1,
### then a tornado is indicated at 15:00,16:00,17:00

time_window = 1 # hours

### HYPERPARAMETER valid_radius in km
### if a tornado occurs within valid_radius km of a station
### a tornado is indicated for that station within the time_window.

### NOTE: If we want improvements, we can draw a "spherical" lin
# between the starting and end lat,lon of a tornado.
### If this line crosses a valid radius of a station, 
### it will be counted. This ASSUMES TORNADO TAKES STRAIGHT LINE.

val_radius = 50 # km

# rename some columns 

rename = {
            'BEGIN_DATE':'TORNADO_BEGIN_DATE',
            'END_DATE':'TORNADO_END_DATE',
            'BEGIN_TIME': 'TORNADO_BEGIN_TIME',
            'END_TIME':'TORNADO_END_TIME',
            'BEGIN_LAT' : 'TORNADO_BEGIN_LAT',
            'END_LAT' : 'TORNADO_END_LAT',
            'BEGIN_LON' : 'TORNADO_BEGIN_LON',
            'END_LON' : 'TORNADO_END_LON',
            'LATITUDE' : 'STATION_LAT',
            'LONGITUDE' : 'STATION_LON',
            'TIME': 'STATION_TIME'
        }

data.rename(columns=rename,inplace=True)


False

In [41]:
# Tornado starting distance from station
def initial_tornado_to_station_distance(pd_row):
    station_lat_lon = pd_row['STATION_LAT'],pd_row['STATION_LON']
    initial_tornado_lat_lon = pd_row['TORNADO_BEGIN_LAT'],pd_row['TORNADO_BEGIN_LON']
    return lat_lon_metric(station_lat_lon,initial_tornado_lat_lon)
def apply_valid_radius(pd_row,valid_radius : float):
    station_lat_lon = pd_row['STATION_LAT'],pd_row['STATION_LON']
    initial_tornado_lat_lon = pd_row['TORNADO_BEGIN_LAT'],pd_row['TORNADO_BEGIN_LON']
    return within_radius(station_lat_lon, initial_tornado_lat_lon,valid_radius)

data['TORNADO_INITIAL_DISTANCE_FROM_STATION'] = data.apply(lambda r : initial_tornado_to_station_distance(r),axis =1)
data[f'TORNADO_INITIAL_DISTANCE_FROM_STATION_WITHIN_{val_radius}_km'] = data.apply(lambda r: apply_valid_radius(r,val_radius),axis =1)

In [42]:
# Apply time_window

def apply_time_window(pd_row, time_window,valid_radius):
    tornado_begin_time = pd_row['TORNADO_BEGIN_TIME']
    station_time = pd_row['STATION_TIME']
    within_radius_boolean = pd_row[f'TORNADO_INITIAL_DISTANCE_FROM_STATION_WITHIN_{valid_radius}_km']
    # type check below because np.nan is a float in this column
    if type(tornado_begin_time) is float and np.isnan(tornado_begin_time):
        return False
    if not within_radius_boolean:
        return False
    # All times are in HOUR:MIN:SECONDS 
    # Capture HOURs
    tornado_hour = int(tornado_begin_time.split(":")[0])
    station_hour = int(station_time.split(":")[0])
    if np.abs(station_hour - tornado_hour)<= time_window:
        return True
    else:
        return False
    

data['TORNADO_OCCURRENCE'] = data.apply(lambda r : apply_time_window(r, time_window,val_radius), axis = 1)

In [43]:
data

,STATION,NAME,STATION_LAT,STATION_LON,ELEVATION,SOURCE,REPORT_TYPE,CALL_SIGN,QUALITY_CONTROL,AA1,...,TORNADO_END_LAT,TORNADO_BEGIN_LON,TORNADO_END_LON,TORNADO_BEGIN_DATE,TORNADO_BEGIN_TIME,TORNADO_END_DATE,TORNADO_END_TIME,TORNADO_INITIAL_DISTANCE_FROM_STATION,TORNADO_INITIAL_DISTANCE_FROM_STATION_WITHIN_50_km,TORNADO_OCCURRENCE
0,72353013967,"OKLAHOMA CITY WILL ROGERS WORLD AIRPORT, OK US",35.38890,-97.60060,391.7,3,SY-MT,OKC,V020,"01,0000,9,5",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False
1,72353013967,"OKLAHOMA CITY WILL ROGERS WORLD AIRPORT, OK US",35.38890,-97.60060,391.7,3,FM-15,OKC,V020,"01,0000,9,5",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False
2,72353013967,"OKLAHOMA CITY WILL ROGERS WORLD AIRPORT, OK US",35.38890,-97.60060,391.7,3,FM-15,OKC,V020,"01,0000,9,5",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False
3,72353013967,"OKLAHOMA CITY WILL ROGERS WORLD AIRPORT, OK US",35.38890,-97.60060,391.7,3,FM-15,OKC,V020,"01,0000,9,5",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False
4,72353013967,"OKLAHOMA CITY WILL ROGERS WORLD AIRPORT, OK US",35.38890,-97.60060,391.7,3,FM-15,OKC,V020,"01,0000,9,5",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1626347,72354403954,"OKLAHOMA CITY WILEY POST AIRPORT, OK US",35.54113,-97.64725,390.2,7,FM-15,KPWA,V020,"01,0000,9,5",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False
1626348,72354403954,"OKLAHOMA CITY WILEY POST AIRPORT, OK US",35.54113,-97.64725,390.2,7,FM-15,KPWA,V020,"01,0000,9,5",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False
1626349,72354403954,"OKLAHOMA CITY WILEY POST AIRPORT, OK US",35.54113,-97.64725,390.2,7,FM-15,KPWA,V020,"01,0000,9,5",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False
1626350,72354403954,"OKLAHOMA CITY WILEY POST AIRPORT, OK US",35.54113,-97.64725,390.2,7,FM-15,KPWA,V020,"01,0000,9,5",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False


In [33]:
data.drop(columns=[f'TORNADO_INITIAL_DISTANCE_FROM_STATION_WITHIN_{val_radius}_km'],inplace=True)


In [23]:
list(data.columns)

['STATION',
 'NAME',
 'STATION_LAT',
 'STATION_LON',
 'ELEVATION',
 'SOURCE',
 'REPORT_TYPE',
 'CALL_SIGN',
 'QUALITY_CONTROL',
 'AA1',
 'AA2',
 'AA3',
 'AA4',
 'AB1',
 'AD1',
 'AE1',
 'AG1',
 'AH1',
 'AH2',
 'AH3',
 'AH4',
 'AH5',
 'AH6',
 'AI1',
 'AI2',
 'AI3',
 'AI4',
 'AI5',
 'AI6',
 'AJ1',
 'AK1',
 'AL1',
 'AM1',
 'AN1',
 'AT1',
 'AT2',
 'AT3',
 'AT4',
 'AT5',
 'AT6',
 'AT7',
 'AT8',
 'AU1',
 'AU2',
 'AU3',
 'AU4',
 'AU5',
 'AW1',
 'AW2',
 'AW3',
 'AW4',
 'AW5',
 'AW6',
 'AW7',
 'AX1',
 'AX2',
 'AX3',
 'AX4',
 'AX5',
 'AX6',
 'CALL_SIGN.1',
 'CIG',
 'DEW',
 'ED1',
 'EQD',
 'GA1',
 'GA2',
 'GA3',
 'GA4',
 'GA5',
 'GA6',
 'GD1',
 'GD2',
 'GD3',
 'GD4',
 'GE1',
 'GF1',
 'GJ1',
 'GK1',
 'GP1',
 'GQ1',
 'GR1',
 'HL1',
 'IA1',
 'KA1',
 'KA2',
 'KA3',
 'KA4',
 'KB1',
 'KB2',
 'KB3',
 'KC1',
 'KC2',
 'KD1',
 'KD2',
 'KE1',
 'KG1',
 'KG2',
 'MA1',
 'MD1',
 'MF1',
 'MG1',
 'MH1',
 'MK1',
 'MV1',
 'MW1',
 'MW2',
 'MW3',
 'MW4',
 'MW5',
 'OC1',
 'OD1',
 'OE1',
 'OE2',
 'OE3',
 'QUALITY_CONTRO

In [ ]:
### NOTE It looks like there are multiple occurrences of the same tornado in STORM EVENTS for single stations.


data[data['TORNADO_OCCURRENCE'] == True][['STATION','STATION_TIME','YEAR-MONTH-DAY','TORNADO_BEGIN_DATE','TORNADO_BEGIN_TIME','TORNADO_INITIAL_DISTANCE_FROM_STATION','TORNADO_OCCURRENCE']]

,STATION,STATION_TIME,YEAR-MONTH-DAY,TORNADO_BEGIN_DATE,TORNADO_BEGIN_TIME,TORNADO_INITIAL_DISTANCE_FROM_STATION,TORNADO_OCCURRENCE
43025,72353013967,15:00:00,2000-10-22,2000-10-22,16:36:00,46.576796,True
43031,72353013967,15:53:00,2000-10-22,2000-10-22,16:36:00,46.576796,True
43037,72353013967,16:00:00,2000-10-22,2000-10-22,16:36:00,46.576796,True
43043,72353013967,16:53:00,2000-10-22,2000-10-22,16:36:00,46.576796,True
43045,72353013967,17:00:00,2000-10-22,2000-10-22,18:14:00,10.369470,True
...,...,...,...,...,...,...,...
1617004,72354403954,21:53:00,2021-10-26,2021-10-26,22:54:00,31.662357,True
1617007,72354403954,22:53:00,2021-10-26,2021-10-26,22:54:00,31.662357,True
1617009,72354403954,22:53:00,2021-10-26,2021-10-26,23:36:00,37.973072,True
1617010,72354403954,23:53:00,2021-10-26,2021-10-26,22:54:00,31.662357,True


In [ ]:
# BEGIN DATA ANALYSIS (BEFORE FEATURE SELECTION)